# 🌐 API & FastAPI Interview Prep

Hands-on exercises. Write the code yourself.

First, install FastAPI:
```
pip install fastapi uvicorn httpx
```

---

## Part 1: REST Fundamentals

### What is a REST API?

An API is a way for programs to talk to each other over HTTP. REST is a set of conventions:

| Method | URL | What it does | Example |
|--------|-----|-------------|--------|
| `GET` | `/jobs` | List all jobs | Fetch the list |
| `GET` | `/jobs/123` | Get one job | Fetch job 123 |
| `POST` | `/jobs` | Create a job | Send JSON body |
| `PUT` | `/jobs/123` | Replace job entirely | Send full object |
| `PATCH` | `/jobs/123` | Update part of job | Send only changed fields |
| `DELETE` | `/jobs/123` | Delete a job | Remove job 123 |

**Status codes:**
- `200` OK — it worked
- `201` Created — new resource made
- `400` Bad Request — you sent garbage
- `401` Unauthorized — who are you?
- `404` Not Found — doesn't exist
- `422` Validation Error — data shape is wrong
- `500` Internal Server Error — server broke

---

### 1.1 — Design an API (pen and paper exercise)

You're building Pasqal's cloud platform. Users submit quantum jobs, check their status, and get results.

**Exercise**: In the cell below, write out (as comments) the endpoints you'd need:
- What method + URL for each action?
- What goes in the request body (for POST)?
- What does the response look like?

Think about: submitting a job, listing jobs, getting one job, cancelling a job.

In [ ]:
# Design your API here (just comments, no code yet):
#
# Submit a job:
#   Method? URL? Body? Response?
# POST /jobs {job defining json}
# List all jobs:
#   Method? URL? Query params? Response?
# GET /jobs
# Get one job:
#   Method? URL? Response?
# GET /jobs/123
# Cancel a job:
#   Method? URL? Response?
# Delete or Patch? / jobs/123


---

## Part 2: FastAPI Basics

### How FastAPI works

FastAPI turns Python functions into API endpoints using decorators:
```python
from fastapi import FastAPI
app = FastAPI()

@app.get("/hello")        # decorator = this function handles GET /hello
def say_hello():           # regular function
    return {"msg": "hi"}   # return a dict → becomes JSON automatically
```

Normally you run it with `uvicorn main:app --reload` and test in a browser.
But in this notebook, we'll use `TestClient` to call our API directly.

---

### 2.1 — Your first endpoint

**Exercise**: Create a FastAPI app with a `GET /` endpoint that returns `{"status": "online"}`.

Then use TestClient to call it and print the response.

In [6]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()

# YOUR CODE: add a GET / endpoint
@app.get("/job_status")
def job_status():
    return {"status": "online" }

# Test it:
client = TestClient(app)
response = client.get("/job_status")
print(response.status_code)  # should be 200
print(response.json())       # should be {"status": "online"}


200
{'status': 'online'}


### 2.2 — Path parameters

Path parameters let you capture parts of the URL:
```python
@app.get("/users/{user_id}")     # {user_id} is a path param
def get_user(user_id: int):       # type hint = auto validation
    return {"user_id": user_id}
```

If someone calls `/users/abc`, FastAPI returns a 422 because `abc` isn't an `int`.

**Exercise**: Create a `GET /jobs/{job_id}` endpoint. Use a simple dict as a fake database.
- If the job exists, return it
- If not, return a 404 error

Hint for 404:
```python
from fastapi import HTTPException
raise HTTPException(status_code=404, detail="Not found")
```

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

app = FastAPI()

# Fake database
fake_db = {
    "job1": {"id": "job1", "name": "bell_state", "status": "completed"},
    "job2": {"id": "job2", "name": "grover", "status": "pending"},
}

# YOUR CODE: GET /jobs/{job_id} — return job or 404


# Test it:
client = TestClient(app)
print(client.get("/jobs/job1").json())        # should return the job
print(client.get("/jobs/nope").status_code)   # should be 404


### 2.3 — Query parameters

Query params are the `?key=value` part of a URL:
```python
@app.get("/items")
def list_items(color: str = None, limit: int = 10):
    # GET /items?color=red&limit=5
    # color = "red", limit = 5
    # GET /items
    # color = None, limit = 10 (defaults)
```

Parameters with defaults are optional. Without defaults = required.

**Exercise**: Create `GET /jobs` that returns all jobs from `fake_db`, with optional filtering:
- `?status=pending` → only pending jobs
- No param → all jobs

Return a list of jobs.

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()

fake_db = {
    "job1": {"id": "job1", "name": "bell_state", "status": "completed"},
    "job2": {"id": "job2", "name": "grover", "status": "pending"},
    "job3": {"id": "job3", "name": "qaoa", "status": "pending"},
}

# YOUR CODE: GET /jobs with optional status filter


# Test it:
client = TestClient(app)
print("All:", client.get("/jobs").json())                    # 3 jobs
print("Pending:", client.get("/jobs?status=pending").json()) # 2 jobs


---

## Part 3: Pydantic & POST Requests

### What is Pydantic?

Pydantic validates data using Python classes with type hints:
```python
from pydantic import BaseModel

class JobCreate(BaseModel):
    name: str              # required
    target: str            # required
    shots: int = 100       # optional, defaults to 100

# FastAPI uses this to validate POST request bodies:
@app.post("/jobs", status_code=201)
def create_job(job: JobCreate):    # FastAPI parses JSON body into this
    print(job.name)                # access fields with dot notation
    print(job.model_dump())        # convert to dict: {"name": ..., "target": ..., "shots": ...}
    return {"id": "new_id", **job.model_dump()}
```

If the request body is missing `name`, FastAPI auto-returns a 422 error. You don't write validation code.

To send a POST with TestClient:
```python
response = client.post("/jobs", json={"name": "test", "target": "qpu"})
```

---

### 3.1 — Create a job via POST

**Exercise**: Build a complete mini API with:
- A Pydantic model `JobCreate` with fields: `name` (str), `target` (str), `shots` (int, default 100)
- `POST /jobs` — creates a job, assigns a UUID, stores it, returns it with status 201
- `GET /jobs/{job_id}` — returns the job or 404
- `GET /jobs` — lists all jobs

Use a plain dict as your database. Use `uuid.uuid4()` for IDs.

In [ ]:
import uuid
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel

app = FastAPI()
db = {}  # our in-memory database

# YOUR CODE: Pydantic model + 3 endpoints


# Test it:
client = TestClient(app)
r1 = client.post("/jobs", json={"name": "bell", "target": "qpu"})
print("Created:", r1.status_code, r1.json())

job_id = r1.json()["id"]
print("Get:", client.get(f"/jobs/{job_id}").json())
print("List:", client.get("/jobs").json())
print("404:", client.get("/jobs/fake").status_code)
print("422:", client.post("/jobs", json={"name": "oops"}).status_code)  # missing target


### 3.2 — PATCH: Update a job's status

PATCH updates part of a resource. For optional fields in Pydantic:
```python
class JobUpdate(BaseModel):
    status: str = None     # all fields optional
    name: str = None
```

To update only provided fields:
```python
update_data = job_update.model_dump(exclude_none=True)  # drops None fields
# Then merge: db[job_id].update(update_data)
```

**Exercise**: Add a `PATCH /jobs/{job_id}` endpoint that updates a job's status.

Test: create a job, then PATCH it to `"completed"`.

In [ ]:
# YOUR CODE: add PATCH endpoint to your app above, or rebuild here


# Test it:
# client = TestClient(app)
# r = client.post("/jobs", json={"name": "test", "target": "emu_mps"})
# job_id = r.json()["id"]
# client.patch(f"/jobs/{job_id}", json={"status": "completed"})
# print(client.get(f"/jobs/{job_id}").json())  # status should be "completed"


### 3.3 — DELETE

```python
@app.delete("/items/{item_id}", status_code=204)
def delete_item(item_id: str):
    del db[item_id]
    return None  # 204 = No Content
```

**Exercise**: Add `DELETE /jobs/{job_id}`. Return 404 if not found, 204 if deleted.

In [ ]:
# YOUR CODE


# Test it:
# r = client.post("/jobs", json={"name": "temp", "target": "qpu"})
# job_id = r.json()["id"]
# print(client.delete(f"/jobs/{job_id}").status_code)  # 204
# print(client.get(f"/jobs/{job_id}").status_code)     # 404


---

## Part 4: Advanced Concepts

### 4.1 — Dependency Injection

Instead of repeating logic (like getting a DB connection) in every endpoint, you inject it:
```python
from fastapi import Depends

def get_db():
    db = connect_to_database()
    try:
        yield db           # give the db to the endpoint
    finally:
        db.close()         # cleanup after the request

@app.get("/items")
def list_items(db = Depends(get_db)):   # FastAPI calls get_db for you
    return db.query("SELECT * FROM items")
```

**Exercise**: Create a dependency `get_job_or_404(job_id: str)` that:
- Looks up the job in the db
- Returns it if found
- Raises HTTPException(404) if not

Use it in both `GET /jobs/{job_id}` and `PATCH /jobs/{job_id}` to avoid repeating the lookup logic.

In [ ]:
from fastapi import FastAPI, HTTPException, Depends
from fastapi.testclient import TestClient

app = FastAPI()
db = {
    "j1": {"id": "j1", "name": "bell", "status": "pending"},
}

# YOUR CODE: dependency function + endpoints that use it


# Test it:
client = TestClient(app)
print(client.get("/jobs/j1").json())       # works
print(client.get("/jobs/nope").status_code) # 404


### 4.2 — Routers

Routers split your app into modules. Each file handles one resource:
```python
# In a real project:
# routers/jobs.py
from fastapi import APIRouter
router = APIRouter(prefix="/jobs", tags=["jobs"])

@router.get("/")
def list_jobs(): ...

@router.get("/{job_id}")
def get_job(job_id: str): ...

# main.py
from fastapi import FastAPI
from routers.jobs import router
app = FastAPI()
app.include_router(router)
```

**Exercise**: Refactor — create a router for jobs and a router for users (just a dummy `GET /users` returning a list).
Include both in the app.

In [ ]:
from fastapi import FastAPI, APIRouter
from fastapi.testclient import TestClient

# YOUR CODE: two routers + app


# Test it:
# client = TestClient(app)
# print(client.get("/jobs").json())
# print(client.get("/users").json())


### 4.3 — Response Models

Control what your API returns (hide internal fields):
```python
class JobResponse(BaseModel):
    id: str
    name: str
    status: str
    # note: no internal fields like "internal_queue_id"

@app.get("/jobs/{job_id}", response_model=JobResponse)
def get_job(job_id: str):
    return db[job_id]  # even if db has extra fields, only JobResponse fields are returned
```

**Exercise**: Create `JobResponse` that only exposes `id`, `name`, `status`, `target`.
Store jobs with an extra `internal_token` field. Verify the response doesn't leak it.

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel

app = FastAPI()

db = {
    "j1": {"id": "j1", "name": "bell", "status": "done", "target": "qpu", "internal_token": "SECRET123"},
}

# YOUR CODE: response model + endpoint


# Test it:
# client = TestClient(app)
# result = client.get("/jobs/j1").json()
# print(result)                              # should NOT contain internal_token
# assert "internal_token" not in result


---

## Part 5: Pasqal Design Challenge

### 5.1 — Build a mini Pasqal Cloud API

Put it all together. Build an API with:

- `POST /jobs` — submit a quantum job (name, target: qpu|emu_mps|emu_sv, shots)
- `GET /jobs` — list jobs, filterable by `?status=` and `?target=`
- `GET /jobs/{job_id}` — get one job
- `PATCH /jobs/{job_id}` — update status
- `DELETE /jobs/{job_id}` — cancel/delete a job

Requirements:
- Use Pydantic models for input AND output
- Validate that `target` is one of `qpu`, `emu_mps`, `emu_sv` (hint: use `Literal` from typing)
- Jobs get a UUID and `created_at` timestamp on creation
- Jobs start with status `queued`

Hint for Literal validation:
```python
from typing import Literal

class JobCreate(BaseModel):
    target: Literal["qpu", "emu_mps", "emu_sv"]  # only these values allowed
```

This is your final boss. Take your time.

In [ ]:
import uuid
from datetime import datetime
from typing import Literal
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel

app = FastAPI()
db = {}

# YOUR CODE: models + all endpoints


In [ ]:
# Test your API:
client = TestClient(app)

# Create jobs
r1 = client.post("/jobs", json={"name": "bell_state", "target": "qpu", "shots": 500})
r2 = client.post("/jobs", json={"name": "grover", "target": "emu_mps"})
print("Created:", r1.status_code, r1.json())

# Invalid target should fail
r3 = client.post("/jobs", json={"name": "bad", "target": "xbox"})
print("Invalid target:", r3.status_code)  # 422

# List and filter
print("All jobs:", client.get("/jobs").json())
print("QPU only:", client.get("/jobs?target=qpu").json())

# Update and delete
job_id = r1.json()["id"]
client.patch(f"/jobs/{job_id}", json={"status": "completed"})
print("Updated:", client.get(f"/jobs/{job_id}").json())

client.delete(f"/jobs/{r2.json()['id']}")
print("After delete:", client.get("/jobs").json())
